# aug21 addendum — B1 entropy/KL series only (~1 h on T4)
**Settings → Accelerator → GPU T4 x2, Internet → On.** Secret: `HF_TOKEN`.
Produces the one missing R3 deliverable: `data/b1_noscale_ckpt_curve.json`
(same JSON shape as `entropy-kl-7b.json`), pushed to the Hub at `aug21/`.
The main run's cell 6 failed silently (`!` swallowed the exit code); this
notebook runs the same command through subprocess and fails loudly.

In [ ]:
# Cell 1 — setup (no vllm needed)
import os, glob, json, subprocess
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'
!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!git checkout analysis/aug21
!pip install -q -e . peft trl bitsandbytes accelerate
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='artifacts/sft-7b', token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-noscale-ckpt', local_dir='artifacts/grpo-7b-noscale', token=os.environ['HF_TOKEN'])
ncs = sorted(glob.glob('artifacts/grpo-7b-noscale/checkpoint-*'), key=lambda x: int(x.rsplit('-', 1)[1]))
print('checkpoints:', [c.rsplit('-', 1)[1] for c in ncs])
assert ncs, 'no checkpoints downloaded'


In [ ]:
# Cell 2 — entropy/KL sweep, LOUD failure (subprocess, not ! magic)
cmd = ['python', '-m', 'connections_rl.eval.entropy_kl',
       '--model', 'Qwen/Qwen2.5-7B-Instruct', '--load-in-4bit',
       '--sft-adapter', 'artifacts/sft-7b',
       '--checkpoints', 'base=base', 'sft=artifacts/sft-7b',
       *[f"ckpt-{c.rsplit('-',1)[1]}={c}" for c in ncs],
       '--puzzles', 'data/splits/puzzles_val.json', '--n', '100',
       '--temperature', '0.9', '--seed', '0',
       '--out', 'results-analysis/aug21/entropy-kl-7b-noscale']
r = subprocess.run(cmd)
assert r.returncode == 0, f'entropy_kl FAILED with exit code {r.returncode} -- do not push partial output'
pts = json.load(open('results-analysis/aug21/entropy-kl-7b-noscale.json'))
assert all(p['n_puzzles'] == 100 for p in pts), 'n_puzzles != 100 somewhere'
for p in pts:
    print(p['name'], p['step'], 'entropy', round(p['entropy_per_token'], 4),
          'KL_sft/seq', round(p['kl_from_sft_per_sequence'], 2),
          'sem', p['semantic_groups_correct'])


In [ ]:
# Cell 3 — map to the exact deliverable path + persist
import shutil
os.makedirs('data', exist_ok=True)
shutil.copyfile('results-analysis/aug21/entropy-kl-7b-noscale.json', 'data/b1_noscale_ckpt_curve.json')
from huggingface_hub import HfApi
api = HfApi()
for src, dst in [('results-analysis/aug21/entropy-kl-7b-noscale.json', 'aug21/entropy-kl-7b-noscale.json'),
                 ('results-analysis/aug21/entropy-kl-7b-noscale.png', 'aug21/entropy-kl-7b-noscale.png'),
                 ('data/b1_noscale_ckpt_curve.json', 'aug21/data/b1_noscale_ckpt_curve.json')]:
    if os.path.exists(src):
        api.upload_file(path_or_fileobj=src, path_in_repo=dst,
                        repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset')
        print('pushed', dst)
print('done -- the agent verifies from aug21/ on the Hub')
